# 02 - Meteorite Bulk Composition with Sulfur

This notebook prepares EF, EC, OC, and CI bulk compositions for the sulfur differentiation workflow. It converts component wt% to molar amounts, normalizes to MgO, applies the existing EF corrections, and exports a table for the embryo-composition step.

`S` denotes **total bulk sulfur**, not a prescribed metal or silicate sulfur amount. The phase partition is calculated later. The four sulfur wt% inputs below are workflow assumptions supplied for this calculation; their literature or dataset provenance should be recorded before scientific interpretation.

**Output:** `notebooks/work_flow_S/Tables/Meteorites_composition.csv`. Run cells in order from the project root or the `notebooks/work_flow_S` directory.

**Mass basis:** The MgO-normalized table remains a plotting/reference product. A separate inventory table stores moles per 100 g of the complete source body; a coverage table records mass outside the modeled components. EF corrections keep the original MgO mass anchor and change the unmodeled remainder. Total S is counted once before phase separation.


In [ ]:
# Raw sulfur wt% from EF/EC/OC/CI meteorite data; recorded inputs, subject to later revision.
PROJECT_ROOT = None             # Repository root; None = detect from current directory
WORKFLOW_DIR = None            # Stage workspace; None = kong_2027_backup/work_flow_S
SULFUR_WT = {'EF': 0.0, 'EC': 2.378344, 'OC': 4.429226, 'CI': 11.520290}
EF_ENRICHMENT_FACTOR = 1.5  # Al2O3/CaO relative to CI
EF_DEPLETION_FACTOR = 0.6   # SiO2/Si relative to original EF
OUTPUT_FILE = None             # MgO-normalized display table
OUTPUT_INVENTORY_FILE = None   # mol per 100 g of complete source body
OUTPUT_COVERAGE_FILE = None    # modeled and unmodeled mass fractions


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
from pathlib import Path
working_dir = Path.cwd()
project_root = (Path(PROJECT_ROOT).expanduser().resolve() if PROJECT_ROOT is not None
                else next((p for p in (working_dir, *working_dir.parents)
                           if (p / "src" / "accrediff").is_dir()), None))
if project_root is None:
    raise RuntimeError("Could not locate the AccreDiff repository root.")
workflow_dir = (Path(WORKFLOW_DIR).expanduser().resolve() if WORKFLOW_DIR is not None
                else project_root / 'kong_2027_backup' / 'work_flow_S')
sys.path.insert(0, str(project_root / 'src'))
import accrediff as ad

# ── Plot style ──────────────────────────────────────────
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "axes.linewidth": 1,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": False,
    "ytick.right": False,
    "figure.dpi": 150
})
Colors = {
    'Earth': "#1f77b4",
    'Mars' : "#d62728",
    'EF'   : "#8c564b",
    'EC'   : "#ff7f0e",
    'OC'   : "#2ca02c",
    'CI'   : "#9467bd",
}

# ── Load constants ───────────────────────────────────────
Elements = ad.constants.Elements.copy()
EF_bulk = ad.constants.EF_bulk.copy()
EC_bulk = ad.constants.EC_bulk.copy()
OC_bulk = ad.constants.OC_bulk.copy()
CI_bulk = ad.constants.CI_bulk.copy()

# Total bulk sulfur in wt% (workflow inputs; provenance to be documented).
sulfur_wt = SULFUR_WT
for name, comp in [('EF', EF_bulk), ('EC', EC_bulk), ('OC', OC_bulk), ('CI', CI_bulk)]:
    comp['S'] = sulfur_wt[name]
# ── Compute molar masses for oxides & metals ─────────────
calc = ad.MolarMassCalculator(mass_table=Elements)
compounds = ['MgO', 'Al2O3', 'CaO', 'FeO', 'NiO', 'SiO2', 'Fe', 'Ni', 'Si', 'O', 'S']
for c in compounds:
    Elements[c] = calc.molar_mass(c)

mantle_items = ['MgO', 'Al2O3', 'CaO', 'FeO', 'NiO', 'SiO2']
core_items   = ['Fe', 'Ni', 'Si', 'O']
bulk_sulfur_items = ['S']
bulk_items   = mantle_items + core_items + bulk_sulfur_items

print("Setup complete.")

## 1. Convert wt% to Molar Amounts

Molar amount is calculated as:

$$n_i = \frac{w_i}{M_i}$$

where $w_i$ is the weight percent and $M_i$ is the molar mass (g/mol).

If a component is absent in a meteorite type, it is set to `NaN` and later replaced by `0`.

In [ ]:
# Convert weight percentages to molar amounts
EF_bulk_molar, EC_bulk_molar, OC_bulk_molar, CI_bulk_molar = {}, {}, {}, {}

for item in bulk_items:
    EF_bulk_molar[item] = EF_bulk[item] / Elements[item] if item in EF_bulk and Elements[item] else np.nan
    EC_bulk_molar[item] = EC_bulk[item] / Elements[item] if item in EC_bulk and Elements[item] else np.nan
    OC_bulk_molar[item] = OC_bulk[item] / Elements[item] if item in OC_bulk and Elements[item] else np.nan
    CI_bulk_molar[item] = CI_bulk[item] / Elements[item] if item in CI_bulk and Elements[item] else np.nan

df_molar = pd.DataFrame({
    'EF': EF_bulk_molar,
    'EC': EC_bulk_molar,
    'OC': OC_bulk_molar,
    'CI': CI_bulk_molar,
}).fillna(0)  # Replace NaN with 0
print("Molar amounts:")
df_molar

## 2. Normalize to MgO

All molar amounts are divided by the MgO molar amount to obtain dimensionless molar ratios relative to Mg:

$$X/\text{MgO} = \frac{n_i}{n_{\text{MgO}}}$$

`NaN` values (missing components) are replaced with `0` after normalization.

In [ ]:
# Normalize to MgO
EF_bulk_mg = {k: v / EF_bulk_molar['MgO'] for k, v in EF_bulk_molar.items()}
EC_bulk_mg = {k: v / EC_bulk_molar['MgO'] for k, v in EC_bulk_molar.items()}
OC_bulk_mg = {k: v / OC_bulk_molar['MgO'] for k, v in OC_bulk_molar.items()}
CI_bulk_mg = {k: v / CI_bulk_molar['MgO'] for k, v in CI_bulk_molar.items()}

df_mg = pd.DataFrame({
    'EF': EF_bulk_mg,
    'EC': EC_bulk_mg,
    'OC': OC_bulk_mg,
    'CI': CI_bulk_mg,
}).fillna(0)  # Replace NaN with 0
print("MgO-normalized molar ratios:")
df_mg

## 3. Apply Enrichment / Depletion Corrections to EF

Enstatite Felsic (EF) meteorites are characterized by:
- **Enrichment** of refractory elements Al and Ca relative to CI (`× 1.5`)
- **Depletion** of Si in both mantle (`SiO2`) and core (`Si`) phases (`× 0.6`)

These corrections are applied directly to the MgO-normalized values `EF_bulk_mg`.

In [ ]:
# Apply enrichment and depletion factors to EF
enrich = EF_ENRICHMENT_FACTOR
poor = EF_DEPLETION_FACTOR

EF_bulk_mg['Al2O3'] = CI_bulk_mg['Al2O3'] * enrich
EF_bulk_mg['CaO']   = CI_bulk_mg['CaO']   * enrich
EF_bulk_mg['SiO2'] *= poor
EF_bulk_mg['Si']   *= poor

print(f"Al2O3 enrichment factor: ×{enrich}")
print(f"CaO   enrichment factor: ×{enrich}")
print(f"SiO2  depletion  factor: ×{poor}")
print(f"Si    depletion  factor: ×{poor}")

pd.DataFrame({'EF_corrected': EF_bulk_mg})

## 4. Composition comparison

The existing spider diagram compares Fe, Ni, and Si in mantle oxides and metal, normalized to MgO and the corresponding CI oxide. Sulfur is a bulk total at this stage and is therefore reported separately as `S/MgO` in the exported table. Earth and Mars sulfur are not inferred from this meteorite input.


In [ ]:
Earth_bulk_mg = ad.constants.Earth_bulk_mg
Mars_bulk_mg   = ad.constants.Mars_bulk_mg
systems = [
    ('Earth', Earth_bulk_mg),
    ('Mars', Mars_bulk_mg),
    ('EF', EF_bulk_mg),
    ('EC', EC_bulk_mg),
    ('OC', OC_bulk_mg),
    ('CI', CI_bulk_mg)
]
# Mantle elements Normalized to MgO and CI
mantle_CI = {}
for name, comp in systems:
    mantle_CI[name] = {i: comp[i] / CI_bulk_mg[i] for i in mantle_items}
# Core elements Normalized to MgO and CI
core_CI = {}
for name, comp in systems:
    core_CI[name] = {i: comp[i] / CI_bulk_mg[j] for i, j in [('Fe', 'FeO'), ('Ni', 'NiO'), ('Si', 'SiO2')]}

Earth_mantle_CI, Earth_core_CI = mantle_CI['Earth'], core_CI['Earth']
Mar_mantle_CI, Mar_core_CI = mantle_CI['Mars'], core_CI['Mars']
EF_mantle_CI, EF_core_CI = mantle_CI['EF'], core_CI['EF']
EC_mantle_CI, EC_core_CI = mantle_CI['EC'], core_CI['EC']
OC_mantle_CI, OC_core_CI = mantle_CI['OC'], core_CI['OC']

In [ ]:
# Elements, keys, and x-axis indices
elements   = ['Al', 'Ca', 'Fe', 'Ni', 'Si']
mantle_keys = ['Al2O3', 'CaO', 'FeO', 'NiO', 'SiO2']
core_keys   = ['Fe', 'Ni', 'Si']
x_idx  = np.arange(len(elements))          # [0,1,2,3,4]
xx_idx = np.array([2, 3, 4])               # Fe, Ni, Si positions

# Build plot datasets: (label, y-values, color, x-positions, marker)
plot_groups = [
    {
        'title'  : 'Mantle',
        'marker' : 's',
        'x_base' : x_idx,
        'data'   : [
            (name, [mantle_CI[name][k] for k in mantle_keys], Colors[name])
            for name in ['Earth', 'Mars', 'EF', 'EC', 'OC']
        ],
    },
    {
        'title'  : 'Core',
        'marker' : 'o',
        'x_base' : xx_idx,
        'data'   : [
            (name, [core_CI[name][k] for k in core_keys], Colors[name])
            for name in ['Earth', 'Mars', 'EF', 'EC', 'OC']
        ],
    },
]

fig, ax = plt.subplots(figsize=(8, 5))
legend_groups = {}

for group in plot_groups:
    offsets = np.linspace(-0.2, 0.2, len(group['data']))
    handles_list = []
    for offset, (label, y, color) in zip(offsets, group['data']):
        sc = ax.scatter(
            group['x_base'] + offset, y,
            label=label, color=color,
            marker=group['marker'], s=65, edgecolor='k', alpha=1
        )
        handles_list.append(sc)
    legend_groups[group['title']] = handles_list

ax.set_yscale('log')
ax.set_xticks(x_idx)
ax.set_xticklabels(elements, fontsize=12)
ax.set_ylabel('Normalized to Mg and CI chondrites', fontsize=12)
ax.axhline(1.0, color='gray', linestyle='--', linewidth=1)
ax.grid(True, which='major', ls='--', lw=0.5, alpha=0.6)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Add grouped legends
labels_list = ['Earth', 'Mars', 'EF', 'EC', 'OC']
leg1 = ax.legend(legend_groups['Mantle'], labels_list, title='Mantle',
                 bbox_to_anchor=(1.0, 1.0), loc='upper left', fontsize=9)
ax.add_artist(leg1)
ax.legend(legend_groups['Core'], labels_list, title='Core',
          bbox_to_anchor=(1.0, 0.55), loc='upper left', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# DataFrame for bulk compositions
df_bulk = pd.DataFrame({
    'EF': EF_bulk_mg,
    'EC': EC_bulk_mg,
    'OC': OC_bulk_mg,
    'CI': CI_bulk_mg
})
# 用0填充df_bulk中的NaN
df_bulk = df_bulk.fillna(0)
# MgO-normalized ratios are for plotting; recover a separate full-body
# inventory using the ORIGINAL MgO wt% as the common mass anchor.
original_mgo_moles = {
    'EF': EF_bulk_molar['MgO'], 'EC': EC_bulk_molar['MgO'],
    'OC': OC_bulk_molar['MgO'], 'CI': CI_bulk_molar['MgO'],
}
df_inventory = df_bulk.mul(pd.Series(original_mgo_moles), axis=1)
df_inventory.index.name = 'component'
df_coverage = pd.DataFrame(index=df_inventory.columns)
df_coverage.index.name = 'source'
df_coverage['tracked_mass_fraction'] = (
    df_inventory.mul(pd.Series({item: Elements[item] for item in bulk_items}), axis=0)
    .sum(axis=0) / 100.0
)
df_coverage['untracked_mass_fraction'] = 1.0 - df_coverage['tracked_mass_fraction']
if not np.isfinite(df_inventory.to_numpy()).all() or (df_inventory < 0).any().any():
    raise ValueError('Endmember inventories must be finite and nonnegative.')
if not df_coverage['untracked_mass_fraction'].between(0.0, 1.0).all():
    raise ValueError('Adjusted endmember composition exceeds complete-body mass.')
print('Modeled mass fractions after the EF correction:')
display(df_coverage)
df_bulk

In [ ]:
# Export only into the sulfur workflow's Tables directory.
from pathlib import Path

output_dir = workflow_dir / 'Tables'
output_dir.mkdir(parents=True, exist_ok=True)
save_file = Path(OUTPUT_FILE) if OUTPUT_FILE is not None else output_dir / 'Meteorites_composition.csv'
df_bulk.to_csv(save_file, index=True)
inventory_file = (Path(OUTPUT_INVENTORY_FILE) if OUTPUT_INVENTORY_FILE is not None
                  else output_dir / 'Meteorites_inventory_per_100g.csv')
coverage_file = (Path(OUTPUT_COVERAGE_FILE) if OUTPUT_COVERAGE_FILE is not None
                 else output_dir / 'Meteorites_mass_coverage.csv')
df_inventory.to_csv(inventory_file, index=True)
df_coverage.to_csv(coverage_file, index=True)

# Check the hand-off contract for the embryo workflow.
exported = pd.read_csv(save_file, index_col=0)
assert list(exported.columns) == ['EF', 'EC', 'OC', 'CI']
assert list(exported.index) == bulk_items
assert np.isfinite(exported.to_numpy()).all()
assert (exported.to_numpy() >= 0).all()
assert np.allclose(exported.loc['MgO'], 1.0)
exported_inventory = pd.read_csv(inventory_file, index_col=0)
exported_coverage = pd.read_csv(coverage_file, index_col=0)
assert exported_inventory.index.equals(exported.index)
assert exported_inventory.columns.equals(exported.columns)
assert np.allclose(exported_inventory.div(exported_inventory.loc['MgO'], axis=1), exported)
assert np.allclose(
    exported_coverage['tracked_mass_fraction'] + exported_coverage['untracked_mass_fraction'], 1.0
)
print(f'Saved and verified: {save_file}, {inventory_file}, {coverage_file}')
